In [2]:
import requests
import pandas as pd
import geopandas as gpd
import json
from pathlib import Path

## Configuration

This pipeline pulls neighbourhood-level socioeconomic indicators for **Porto Alegre** from the IBGE SIDRA API and joins them to the existing neighbourhood geometry.

All indicators are selected for their relevance to the No-Regret Actions workshop framework:

| Indicator | Workshop relevance |
|---|---|
| Population + density | All groups — demand and scale estimation |
| Poverty rate | All groups — equity layer |
| % piped water | Water Efficiency, Flood & Drainage |
| % without formal sewage | Water Efficiency, Urban Green Infrastructure |

In [5]:
MUNICIPALITY_CODE = '4314902'
BATCH_SIZE = 50
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

## 0. Load geometry and extract neighbourhood IDs

In [3]:
gdf = gpd.read_file('output/porto_alegre_neighbourhoods.geojson')
gdf = gdf[['neighbourhood_number', 'neighbourhood_name', 'geometry']]
trg_nb_array = gdf['neighbourhood_number'].unique()
print(f'Loaded {len(gdf)} neighbourhoods for Porto Alegre')
print(f'CRS: {gdf.crs}')
gdf.head()

Loaded 99 neighbourhoods for Porto Alegre
CRS: EPSG:4326


,neighbourhood_number,neighbourhood_name,geometry
0,4314902014,Santo Antônio,"POLYGON ((-51.20355 -30.06426, -51.20615 -30.0..."
1,4314902047,São Geraldo,"POLYGON ((-51.2005 -30.0149, -51.20084 -30.014..."
2,4314902065,São Sebastião,"POLYGON ((-51.14624 -29.99796, -51.14618 -29.9..."
3,4314902066,Humaitá,"POLYGON ((-51.19085 -29.97681, -51.19101 -29.9..."
4,4314902030,Espírito Santo,"POLYGON ((-51.2194 -30.15242, -51.21973 -30.15..."


## Helper functions

In [6]:
def fetch_batched(url_template, nb_array, batch_size=BATCH_SIZE):
    """Fetch IBGE SIDRA data in batches, return combined normalised DataFrame."""
    all_data = []
    n_batches = (len(nb_array) - 1) // batch_size + 1
    for i in range(0, len(nb_array), batch_size):
        batch = nb_array[i:i + batch_size]
        nb_ids = ','.join(batch)
        url = url_template.format(nb_ids=nb_ids)
        response = requests.get(url)
        if response.status_code == 200:
            all_data.extend(response.json())
            print(f'  Batch {i // batch_size + 1}/{n_batches} ({len(batch)} neighbourhoods)')
        else:
            print(f'  ERROR batch {i // batch_size + 1}: HTTP {response.status_code}')
    return pd.json_normalize(all_data)


def expand_resultados(df):
    """Explode resultados -> series into a flat DataFrame."""
    df = df.explode('resultados').reset_index(drop=True)
    res = pd.json_normalize(df['resultados'])
    df = pd.concat([df.drop('resultados', axis=1).reset_index(drop=True), res], axis=1)
    df = df.explode('series').reset_index(drop=True)
    ser = pd.json_normalize(df['series'])
    df = pd.concat([df.drop('series', axis=1).reset_index(drop=True), ser], axis=1)
    return df


def get_serie_col(df):
    """Return the most recent 'serie.YYYY' column present."""
    serie_cols = sorted([c for c in df.columns if c.startswith('serie.')], reverse=True)
    if not serie_cols:
        raise ValueError('No serie columns found in DataFrame')
    col = serie_cols[0]
    print(f'  Using data column: {col}')
    return col


def extract_classification(classificacoes_list, cls_id):
    """Extract category name and ID for a given classification ID."""
    for item in classificacoes_list:
        if str(item['id']) == str(cls_id):
            cat = item['categoria']
            if isinstance(cat, dict):
                k, v = list(cat.items())[0]
                return str(k), v
    return None, None

## 1. Population

**Agregado 3107** — *População residente, por sexo, situação e grupos de idade*
Variable 93 (população residente), Total situation, Total sex, Total age group.

Used by all workshop groups for demand estimation and density calculation.

In [7]:
print('Fetching population data...')
url_tpl = (
    'https://servicodados.ibge.gov.br/api/v3/agregados/3107/periodos/-1'
    '/variaveis/93?localidades=N102[{nb_ids}]&classificacao=1[0]|2[0]|58[0]'
)
df_pop_raw = fetch_batched(url_tpl, trg_nb_array)
df_pop = expand_resultados(df_pop_raw)

serie_col = get_serie_col(df_pop)
df_pop[serie_col] = pd.to_numeric(df_pop[serie_col], errors='coerce')

population = df_pop[['localidade.id', 'localidade.nome', serie_col]].copy()
population.columns = ['neighbourhood_number', 'neighbourhood_name', 'population_total']
population = population.dropna(subset=['population_total'])

print(f'Population data: {len(population)} neighbourhoods')
population.head()

Fetching population data...
  Batch 1/2 (50 neighbourhoods)
  Batch 2/2 (44 neighbourhoods)
  Using data column: serie.2010
Population data: 75 neighbourhoods


,neighbourhood_number,neighbourhood_name,population_total
0,4314902014,Santo Antônio - Porto Alegre (RS),13161.0
1,4314902047,São Geraldo - Porto Alegre (RS),8292.0
2,4314902065,São Sebastião - Porto Alegre (RS),6511.0
3,4314902066,Humaitá - Porto Alegre (RS),11502.0
4,4314902030,Espírito Santo - Porto Alegre (RS),5606.0


## 2. Poverty Rate

**Agregado 3261** — *Domicílios particulares permanentes, por classes de rendimento nominal mensal domiciliar per capita*
Classification 386: income per capita in minimum wages.

Poverty threshold: households earning <= 1/2 minimum wage per capita (categories 9681 + 9682 + 9692 = no income).

This is the primary equity layer used by all six workshop groups.

In [8]:
# Income classification IDs (classification 386)
# 9680  = Total
# 9681  = Ate 1/4 de salario minimo          <- poor
# 9682  = Mais de 1/4 a 1/2 salario minimo   <- poor
# 9683  = Mais de 1/2 a 1 salario minimo
# 9684  = Mais de 1 a 2 salarios minimos
# 9685  = Mais de 2 a 3 salarios minimos
# 9686  = Mais de 3 a 5 salarios minimos
# 11780 = Mais de 5 salarios minimos
# 9692  = Sem rendimento                     <- poor
POOR_INCOME_IDS = {'9681', '9682', '9692'}

print('Fetching poverty/income data...')
url_tpl = (
    'https://servicodados.ibge.gov.br/api/v3/agregados/3261/periodos/-1'
    '/variaveis/96?localidades=N102[{nb_ids}]&classificacao=386[all]'
)
df_inc_raw = fetch_batched(url_tpl, trg_nb_array)
df_inc = expand_resultados(df_inc_raw)

serie_col = get_serie_col(df_inc)
df_inc[serie_col] = pd.to_numeric(df_inc[serie_col], errors='coerce')

df_inc['income_class_id'] = df_inc['classificacoes'].apply(
    lambda x: extract_classification(x, 386)[0]
)

total_hh = (
    df_inc[df_inc['income_class_id'] == '9680']
    [['localidade.id', 'localidade.nome', serie_col]]
    .copy()
)
total_hh.columns = ['neighbourhood_number', 'neighbourhood_name', 'household_total']

poor_hh = (
    df_inc[df_inc['income_class_id'].isin(POOR_INCOME_IDS)]
    .groupby('localidade.id')[serie_col]
    .sum()
    .reset_index()
)
poor_hh.columns = ['neighbourhood_number', 'poor_households']

poverty = total_hh.merge(poor_hh, on='neighbourhood_number', how='left')
poverty['poor_households'] = poverty['poor_households'].fillna(0)
poverty['poverty_rate'] = (poverty['poor_households'] / poverty['household_total']).round(4)

print(f'Poverty data: {len(poverty)} neighbourhoods')
poverty.head()

Fetching poverty/income data...
  Batch 1/2 (50 neighbourhoods)
  Batch 2/2 (44 neighbourhoods)
  Using data column: serie.2010
Poverty data: 94 neighbourhoods


,neighbourhood_number,neighbourhood_name,household_total,poor_households,poverty_rate
0,4314902014,Santo Antônio - Porto Alegre (RS),5326.0,324.0,0.0608
1,4314902047,São Geraldo - Porto Alegre (RS),3331.0,88.0,0.0264
2,4314902065,São Sebastião - Porto Alegre (RS),2482.0,111.0,0.0447
3,4314902066,Humaitá - Porto Alegre (RS),4165.0,446.0,0.1071
4,4314902030,Espírito Santo - Porto Alegre (RS),1830.0,117.0,0.0639


## 2b. Income Quintile Distribution

Derived from the same `df_inc` fetch above — no additional API call needed.
We extract the count of households in each income bracket to give each group
a full picture of the income landscape, not just the poverty threshold.

| Column | Bracket (per capita, minimum wages) |
|---|---|
| `hh_inc_q1` | ≤ ¼ min wage |
| `hh_inc_q2` | > ¼ to ½ min wage |
| `hh_inc_q3` | > ½ to 1 min wage |
| `hh_inc_q4` | > 1 to 2 min wages |
| `hh_inc_q5` | > 2 min wages |
| `hh_inc_none` | No income |
| `pct_low_income` | Households earning ≤ 1 min wage (q1 + q2 + q3 + none) |
| `pct_high_income` | Households earning > 2 min wages (q5) |

In [ ]:
# Income bracket category IDs (classification 386)
INCOME_BRACKET_MAP = {
    '9681':  'hh_inc_q1',    # <= 1/4 min wage
    '9682':  'hh_inc_q2',    # > 1/4 to 1/2
    '9683':  'hh_inc_q3',    # > 1/2 to 1
    '9684':  'hh_inc_q4',    # > 1 to 2
    '9685':  'hh_inc_q5a',   # > 2 to 3
    '9686':  'hh_inc_q5b',   # > 3 to 5
    '11780': 'hh_inc_q5c',   # > 5
    '9692':  'hh_inc_none',  # No income
}

# Re-use df_inc already fetched in section 2
df_inc_brackets = df_inc[df_inc['income_class_id'].isin(INCOME_BRACKET_MAP.keys())].copy()
df_inc_brackets['bracket_col'] = df_inc_brackets['income_class_id'].map(INCOME_BRACKET_MAP)

# serie_col is already set from section 2 — reuse it
pivot_income = df_inc_brackets.pivot_table(
    index='localidade.id',
    columns='bracket_col',
    values=serie_col,
    aggfunc='sum'
).reset_index()
pivot_income.columns.name = None
pivot_income = pivot_income.rename(columns={'localidade.id': 'neighbourhood_number'})

# Merge in total households for percentage calculations
pivot_income = pivot_income.merge(
    poverty[['neighbourhood_number', 'household_total']], on='neighbourhood_number', how='left'
)

# Consolidate q5 sub-brackets
pivot_income['hh_inc_q5'] = (
    pivot_income.get('hh_inc_q5a', 0).fillna(0)
    + pivot_income.get('hh_inc_q5b', 0).fillna(0)
    + pivot_income.get('hh_inc_q5c', 0).fillna(0)
)

# Summary percentages
low_inc_cols = ['hh_inc_q1', 'hh_inc_q2', 'hh_inc_q3', 'hh_inc_none']
pivot_income['pct_low_income'] = (
    pivot_income[low_inc_cols].fillna(0).sum(axis=1) / pivot_income['household_total']
).round(4)
pivot_income['pct_high_income'] = (
    pivot_income['hh_inc_q5'] / pivot_income['household_total']
).round(4)

income_dist = pivot_income[[
    'neighbourhood_number',
    'hh_inc_q1', 'hh_inc_q2', 'hh_inc_q3', 'hh_inc_q4', 'hh_inc_q5',
    'hh_inc_none', 'pct_low_income', 'pct_high_income'
]].copy()

print(f'Income distribution: {len(income_dist)} neighbourhoods')
income_dist.head()

## 3. Water Supply

**Agregado 1436** — *Domicilios particulares permanentes, por forma de abastecimento de agua*
Classification 61: water supply type.
Category 92853 = Rede geral (formal piped network).

Relevant to Water Efficiency and Flood & Drainage groups.
Low piped-water coverage correlates with informal settlements and flood-exposed areas.

In [9]:
print('Fetching water supply data...')
url_tpl = (
    'https://servicodados.ibge.gov.br/api/v3/agregados/1436/periodos/-1'
    '/variaveis/96?localidades=N102[{nb_ids}]&classificacao=1[0]|61[0,92853]'
)
df_water_raw = fetch_batched(url_tpl, trg_nb_array)
df_water = expand_resultados(df_water_raw)

serie_col = get_serie_col(df_water)
df_water[serie_col] = pd.to_numeric(df_water[serie_col], errors='coerce')

df_water['water_supply_id'] = df_water['classificacoes'].apply(
    lambda x: extract_classification(x, 61)[0]
)

pivot_water = df_water.pivot_table(
    index='localidade.id',
    columns='water_supply_id',
    values=serie_col,
    aggfunc='max'
).reset_index()

pivot_water.columns = ['neighbourhood_number'] + [
    f'water_cls_{c}' for c in pivot_water.columns[1:]
]

water = pivot_water[['neighbourhood_number', 'water_cls_0', 'water_cls_92853']].copy()
water.columns = ['neighbourhood_number', 'hh_total_water', 'hh_piped_water']
water['pct_piped_water'] = (water['hh_piped_water'] / water['hh_total_water']).round(4)

print(f'Water supply data: {len(water)} neighbourhoods')
water.head()

Fetching water supply data...
  Batch 1/2 (50 neighbourhoods)
  Batch 2/2 (44 neighbourhoods)
  Using data column: serie.2000
Water supply data: 74 neighbourhoods


,neighbourhood_number,hh_total_water,hh_piped_water,pct_piped_water
0,4314902001,4096.0,4088.0,0.9980
1,4314902002,745.0,710.0,0.9530
2,4314902003,7821.0,7820.0,0.9999
3,4314902004,11495.0,11486.0,0.9992
4,4314902005,450.0,450.0,1.0000


## 4. Sewage Access

**Agregado 1437** — *Domicilios particulares permanentes, por tipo de esgotamento sanitario*
Classification 62: sanitation type.

Run the metadata cell first to confirm category IDs, then fetch the data.

In [10]:
# Inspect available sewage categories
metadata_url = 'https://servicodados.ibge.gov.br/api/v3/agregados/1437/metadados'
response = requests.get(metadata_url)
if response.status_code == 200:
    metadata = response.json()
    for cls in metadata.get('classificacoes', []):
        print(f"Classification {cls['id']}: {cls['nome']}")
        for cat in cls['categorias']:
            print(f"  {cat['id']:>8} | {cat['nome']}")
else:
    print(f'Metadata fetch failed: {response.status_code}')

Classification 1: Situação do domicílio
         0 | Total
         1 | Urbana
         2 | Rural
Classification 11558: Tipo de esgotamento sanitário
         0 | Total
     92855 | Rede geral de esgoto ou pluvial
     92856 | Fossa séptica
     92857 | Fossa rudimentar
     92858 | Vala
     92859 | Rio, lago ou mar
     92860 | Outro escoadouro
     92861 | Não tinham banheiro nem sanitário


In [11]:
# Update FORMAL_SEWAGE_IDS based on metadata output above
# Classification 11558: Tipo de esgotamento sanitário
# 92855 = Rede geral de esgoto ou pluvial (formal sewage)
FORMAL_SEWAGE_IDS = ['92855']

print('Fetching sewage data...')
url_tpl = (
    'https://servicodados.ibge.gov.br/api/v3/agregados/1437/periodos/-1'
    '/variaveis/96?localidades=N102[{nb_ids}]&classificacao=1[0]|11558[all]'
)
df_sew_raw = fetch_batched(url_tpl, trg_nb_array)
df_sew = expand_resultados(df_sew_raw)

serie_col = get_serie_col(df_sew)
df_sew[serie_col] = pd.to_numeric(df_sew[serie_col], errors='coerce')

df_sew['sewage_id'] = df_sew['classificacoes'].apply(
    lambda x: extract_classification(x, 11558)[0]
)

total_sew = (
    df_sew[df_sew['sewage_id'] == '0']
    [['localidade.id', serie_col]]
    .copy()
)
total_sew.columns = ['neighbourhood_number', 'hh_total_sewage']

formal_sew = (
    df_sew[df_sew['sewage_id'].isin(FORMAL_SEWAGE_IDS)]
    .groupby('localidade.id')[serie_col]
    .sum()
    .reset_index()
)
formal_sew.columns = ['neighbourhood_number', 'hh_formal_sewage']

sewage = total_sew.merge(formal_sew, on='neighbourhood_number', how='left')
sewage['hh_formal_sewage'] = sewage['hh_formal_sewage'].fillna(0)
sewage['pct_formal_sewage'] = (sewage['hh_formal_sewage'] / sewage['hh_total_sewage']).round(4)
sewage['pct_no_formal_sewage'] = (1 - sewage['pct_formal_sewage']).round(4)

print(f'Sewage data: {len(sewage)} neighbourhoods')
sewage.head()

Fetching sewage data...
  ERROR batch 1: HTTP 500
  ERROR batch 2: HTTP 500


KeyError: 'resultados'

## 5. Aglomerados Subnormais (Informal Settlements)

IBGE maps informal settlements — *vilas* and *favelas* — as a dedicated polygon layer
in the *Aglomerados Subnormais* dataset (separate from census tracts).
This is the highest-value equity layer for the Flood & Drainage,
Urban Green Infrastructure, and Active Travel workshop groups.

**Download the shapefile first:**
[ibge.gov.br/geociencias — Aglomerados Subnormais](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/tipologias-do-territorio/15788-aglomerados-subnormais.html)

Save to: `output/aglomerados_subnormais_RS.shp` (filter to Rio Grande do Sul or load the national file).

This cell derives two indicators per neighbourhood:
- `n_agsub` — count of informal settlement polygons whose centroid falls within the neighbourhood
- `pct_area_agsub` — share of neighbourhood area covered by informal settlement polygons

In [ ]:
from pathlib import Path

AGSUB_PATH = Path('output/aglomerados_subnormais_RS.shp')

if not AGSUB_PATH.exists():
    print(
        'Aglomerados Subnormais shapefile not found.\n'
        'Download from IBGE and save to: output/aglomerados_subnormais_RS.shp\n'
        'Skipping — agsub columns will be NaN in the output.'
    )
    agsub_indicators = gdf[['neighbourhood_number']].copy()
    agsub_indicators['n_agsub'] = None
    agsub_indicators['pct_area_agsub'] = None
else:
    # Load and reproject to match neighbourhood CRS
    agsub = gpd.read_file(AGSUB_PATH)
    if agsub.crs != gdf.crs:
        agsub = agsub.to_crs(gdf.crs)

    # Filter to Porto Alegre municipality (CD_MUN = 4314902) if column present
    mun_col = next((c for c in agsub.columns if 'MUN' in c.upper()), None)
    if mun_col:
        agsub = agsub[agsub[mun_col].astype(str).str.startswith('4314902')]
        print(f'Filtered to {len(agsub)} informal settlement polygons in Porto Alegre')

    # Project both layers to UTM 22S for area calculations
    gdf_proj = gdf.to_crs('EPSG:32722')
    agsub_proj = agsub.to_crs('EPSG:32722')

    # Count of agsub centroids per neighbourhood
    agsub_centroids = agsub_proj.copy()
    agsub_centroids['geometry'] = agsub_proj.geometry.centroid
    joined_count = gpd.sjoin(agsub_centroids, gdf_proj[['neighbourhood_number', 'geometry']],
                             how='left', predicate='within')
    n_agsub = (
        joined_count.groupby('neighbourhood_number')
        .size()
        .reset_index(name='n_agsub')
    )

    # Intersection area per neighbourhood
    intersection = gpd.overlay(agsub_proj[['geometry']], gdf_proj[['neighbourhood_number', 'geometry']],
                                how='intersection')
    intersection['agsub_area_m2'] = intersection.geometry.area
    nb_agsub_area = intersection.groupby('neighbourhood_number')['agsub_area_m2'].sum().reset_index()

    nb_areas = gdf_proj[['neighbourhood_number']].copy()
    nb_areas['nb_area_m2'] = gdf_proj.geometry.area

    pct_area = nb_areas.merge(nb_agsub_area, on='neighbourhood_number', how='left')
    pct_area['pct_area_agsub'] = (
        (pct_area['agsub_area_m2'] / pct_area['nb_area_m2']).fillna(0).round(4)
    )

    agsub_indicators = (
        gdf[['neighbourhood_number']]
        .merge(n_agsub, on='neighbourhood_number', how='left')
        .merge(pct_area[['neighbourhood_number', 'pct_area_agsub']], on='neighbourhood_number', how='left')
    )
    agsub_indicators['n_agsub'] = agsub_indicators['n_agsub'].fillna(0).astype(int)
    agsub_indicators['pct_area_agsub'] = agsub_indicators['pct_area_agsub'].fillna(0)

    print(f'Agsub indicators: {len(agsub_indicators)} neighbourhoods')
    print(f"Neighbourhoods with any informal settlement: {(agsub_indicators['n_agsub'] > 0).sum()}")
    agsub_indicators.head()

## 5. Join all indicators to geometry

In [ ]:
indicators = (
    population[['neighbourhood_number', 'population_total']]
    .merge(
        poverty[['neighbourhood_number', 'household_total', 'poor_households', 'poverty_rate']],
        on='neighbourhood_number', how='left'
    )
    .merge(
        income_dist[[
            'neighbourhood_number',
            'hh_inc_q1', 'hh_inc_q2', 'hh_inc_q3', 'hh_inc_q4', 'hh_inc_q5',
            'hh_inc_none', 'pct_low_income', 'pct_high_income'
        ]],
        on='neighbourhood_number', how='left'
    )
    .merge(
        water[['neighbourhood_number', 'pct_piped_water']],
        on='neighbourhood_number', how='left'
    )
    .merge(
        sewage[['neighbourhood_number', 'pct_formal_sewage', 'pct_no_formal_sewage']],
        on='neighbourhood_number', how='left'
    )
    .merge(
        agsub_indicators[['neighbourhood_number', 'n_agsub', 'pct_area_agsub']],
        on='neighbourhood_number', how='left'
    )
)

print(f'Indicator table: {len(indicators)} rows, {len(indicators.columns)} columns')
print(indicators.isnull().sum())
indicators.head()

In [ ]:
gdf_out = gdf.merge(indicators, on='neighbourhood_number', how='left')

if gdf_out.crs is None:
    gdf_out = gdf_out.set_crs('EPSG:4326')

gdf_projected = gdf_out.to_crs('EPSG:32722')  # UTM Zone 22S - Porto Alegre
gdf_out['area_km2'] = (gdf_projected.geometry.area / 1e6).round(4)
gdf_out['pop_density_km2'] = (gdf_out['population_total'] / gdf_out['area_km2']).round(1)

print(f'Output GeoDataFrame: {len(gdf_out)} features')
print(gdf_out.columns.tolist())
gdf_out.drop(columns='geometry').head()

## 6. Output

In [ ]:
out_path = OUTPUT_DIR / 'porto_alegre_indicators.geojson'
gdf_out.to_file(out_path, driver='GeoJSON')
print(f'Saved: {out_path}')

print('\n--- Summary ---')
print(f"Total population:        {gdf_out['population_total'].sum():,.0f}")
print(f"Total households:        {gdf_out['household_total'].sum():,.0f}")
print(f"Avg poverty rate:        {gdf_out['poverty_rate'].mean():.1%}")
print(f"Avg pct piped water:     {gdf_out['pct_piped_water'].mean():.1%}")
print(f"Avg pct no formal sewage:{gdf_out['pct_no_formal_sewage'].mean():.1%}")
print(f"Avg pop density:         {gdf_out['pop_density_km2'].mean():,.0f} people/km2")

## Indicator reference

| Column | Source | Unit | Notes |
|---|---|---|---|
| `population_total` | IBGE Agg 3107, Var 93 | People | Total resident population |
| `household_total` | IBGE Agg 3261, Var 96 | Households | Permanent private dwellings |
| `poor_households` | IBGE Agg 3261, Cls 386 | Households | Per capita income ≤ ½ min wage + no income |
| `poverty_rate` | Derived | 0–1 | poor_households / household_total |
| `hh_inc_q1` | IBGE Agg 3261, Cls 386 | Households | ≤ ¼ min wage per capita |
| `hh_inc_q2` | IBGE Agg 3261, Cls 386 | Households | > ¼ to ½ min wage per capita |
| `hh_inc_q3` | IBGE Agg 3261, Cls 386 | Households | > ½ to 1 min wage per capita |
| `hh_inc_q4` | IBGE Agg 3261, Cls 386 | Households | > 1 to 2 min wages per capita |
| `hh_inc_q5` | IBGE Agg 3261, Cls 386 | Households | > 2 min wages per capita |
| `hh_inc_none` | IBGE Agg 3261, Cls 386 | Households | No declared income |
| `pct_low_income` | Derived | 0–1 | Households earning ≤ 1 min wage (q1+q2+q3+none) |
| `pct_high_income` | Derived | 0–1 | Households earning > 2 min wages (q5) |
| `pct_piped_water` | IBGE Agg 1436, Cls 61 | 0–1 | Households on formal water network |
| `pct_formal_sewage` | IBGE Agg 1437, Cls 62 | 0–1 | Households on formal sewage network |
| `pct_no_formal_sewage` | Derived | 0–1 | 1 − pct_formal_sewage |
| `n_agsub` | IBGE Aglomerados Subnormais | Count | Informal settlement polygons in neighbourhood |
| `pct_area_agsub` | Derived | 0–1 | Share of neighbourhood area covered by informal settlements |
| `area_km2` | Geometry | km² | Neighbourhood area in UTM Zone 22S |
| `pop_density_km2` | Derived | people/km² | population_total / area_km2 |